In [1]:
# baseline model
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

c:\Users\gupta\OneDrive\Desktop\mlops-mini-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df=pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])

In [3]:
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:
# define the preprocessing
nltk.download('wordnet')
nltk.download('stopwords')
def lematization(text):
    lemmatizer=WordNetLemmatizer()
    text=text.split()
    text=[lemmatizer.lemmatize(y) for y in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words=set(stopwords.words('english'))
    Text=[i for i in str(text).split() if i not in stop_words]
    return " ".join(Text)

def removing_numbers(text):
    text=''.join([i for i in text if not i.isdigit()])
    return text

def lower_case(text):
    text=text.split()
    text=[y.lower() for y in text]
    return " ".join(text)
    
def removing_punctuations(text):
    punctuations =  r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""
    # raw string avoids invalid escape warnings
    text= re.sub('[%s]' % re.escape(punctuations), '', text)

    # remove extra whitespace
    text=re.sub(r'\s+', ' ', text)
    text=' '.join(text.split())
    return text.strip()

def removing_urls(text):
    url_pattern=re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def remove_small_sentence(df):
    for i in range(len(df)):
        if len(df.text.iloc[i].split())<3:
            df.text.iloc[i]=np.nan

def normalize_text(df):
    df.content=df.content.apply(lower_case)
    df.content=df.content.apply(remove_stop_words)
    df.content=df.content.apply( removing_numbers)
    df.content=df.content.apply( removing_punctuations)
    df.content=df.content.apply(removing_urls)
    df.content=df.content.apply(lematization)
    return df


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gupta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
df=normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [6]:
df.sentiment.value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [7]:
x=df.sentiment.isin(['happiness', 'sadness'])
df=df[x]

In [8]:
df['sentiment']=df['sentiment'].replace({'happiness':1, 'sadness':0})
df.head()

C:\Users\gupta\AppData\Local\Temp\ipykernel_2812\4003209269.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'happiness':1, 'sadness':0})


,sentiment,content
1,0,layin n bed headache ughhhhwaitin call
2,0,funeral ceremonygloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [9]:
# apply the Countvectorizer
vectorizer=CountVectorizer(max_features=1000)
X=vectorizer.fit_transform(df['content'])
y=df['sentiment']

In [10]:
# split data into train, test
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)

In [11]:
#
import dagshub

dagshub.init(repo_owner='guptatannu538', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/guptatannu538/mlops-mini-project.mlflow')
mlflow.set_experiment('Logistic Regression Baseline')


Accessing as guptatannu538

Initialized MLflow to track repo "guptatannu538/mlops-mini-project"

Repository guptatannu538/mlops-mini-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/402ab37d00994076bd2a397e7fe5769f', creation_time=1784529838989, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1784529838989, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [12]:
import mlflow
with mlflow.start_run():
    # Log preprocessing parameters
    mlflow.log_param('vectorize', 'Bag of Words')
    mlflow.log_param('num_features', 1000)
    mlflow.log_param('test_size', 0.2)

    # Model building and training
    model=LogisticRegression()
    model.fit(X_train, y_train)
    
    # Log model parameters
    mlflow.log_param('model', 'Logistic Regression')

    # Model evaluation
    y_pred=model.predict(X_test)
    accuracy=accuracy_score(y_test, y_pred)
    precision=precision_score(y_test, y_pred)
    recall=recall_score(y_test, y_pred)
    f1=f1_score(y_test, y_pred)

    # Log evaluation metrics
    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('f1', f1)

    # Log model
    mlflow.sklearn.log_model(model, 'model')

    # Save and log the notebook
    import os
    notebook_path='exp1_baseline_model.ipynb'
    os.system(f'jupyter nbconvert --to notebook --execute --inplace {notebook_path}')
    mlflow.log_artifact(notebook_path)

    # print the results for verification
    print(f'Accuracy', accuracy)
    print(f'Precision', precision)
    print(f'recall', recall)
    print(f'f1', f1)



2026/07/20 13:03:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy 0.7783132530120482
Precision 0.7650429799426934
recall 0.7891625615763547
f1 0.7769156159068865
🏃 View run worried-auk-557 at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/0/runs/a0d9ec6f02eb4e528dfee73ad498902d
🧪 View experiment at: https://dagshub.com/guptatannu538/mlops-mini-project.mlflow/#/experiments/0
